<a href="https://colab.research.google.com/github/Neilaurel/Aurel/blob/main/quickstarts/Get_started_managed_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Agents API: Build managed agents with the Interactions API

<a class="tfo-notebook-buttons" target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_managed_agents.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The [Interactions API](https://ai.google.dev/gemini-api/docs/interactions) provides a unified interface for working with Gemini models and agents. The [Getting Started notebook](./Get_started_interactions_api.ipynb) covers how to use it with standard Gemini **models** for text generation, multi-turn conversations, and tool use.

This notebook focuses on something different: **managed agents** with the `antigravity-preview-05-2026` agent.

### `agent=` vs `model=`

When you call the Interactions API, you choose between two modes:

| Parameter | What runs | Best for |
|-----------|-----------|----------|
| `model="gemini-..."` | A standard Gemini model | Text generation, structured output, function calling |
| `agent="antigravity-preview-05-2026"` | A **managed agent** in a sandboxed Linux environment | Autonomous tasks: code execution, web research, file management |

With `model=`, you get a stateless LLM call (see the [Getting Started notebook](./Get_started_interactions_api.ipynb)). With `agent=`, you spin up an autonomous agent that can **reason, plan, write and execute code, browse the web, and manage files** — all inside a secure sandbox, without you writing any orchestration logic.

This notebook walks you through the agent mode step by step:

1. **Simple questions** — use the agent like an LLM (it works, but it's overkill!)
2. **Multi-turn conversations** — persistent sandbox = built-in memory
3. **Using tools** — code execution, web search, file operations
4. **Loading data into the sandbox** — inject files before the agent starts
5. **Creating reusable custom agents** — bundle instructions, skills, and environment

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [ ]:
%pip install -U -q "google-genai>=2.9.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.4.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.4.0 which is incompatible.


### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key or you aren't sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK, now you only need to initialize a client with you API key.

In [ ]:
import uuid
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=GEMINI_API_KEY)

# The default managed agent.
AGENT = "antigravity-preview-05-2026"

# Generate a unique suffix for this notebook session to prevent agent ID conflicts
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

print("Client ready!")

Client ready!


## 1. Simple questions — the agent as an LLM

The simplest way to use a managed agent is to ask it a question, just like you'd call a standard Gemini model. Pass `agent="antigravity-preview-05-2026"` and `environment="remote"` to create a fresh Linux sandbox for the agent.

This works, but it's a bit like driving a Formula 1 car to the grocery store — the agent has code execution, web search, and file management capabilities that are all sitting idle for a simple factual question.

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What is the capital of France?",
    environment="remote",
)

Markdown(interaction.output_text)

/tmp/ipykernel_2790/271594579.py:1: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = client.interactions.create(


The capital of France is Paris.

In [ ]:
# The response also includes metadata about the agent's sandbox.
print(f"Status:         {interaction.status}")
print(f"Interaction ID: {interaction.id}")
print(f"Environment ID: {interaction.environment_id}")

Status:         completed
Interaction ID: v1_Chc1SlFNYXVmUE51YXNqckVQM09laGlBYxIXNUpRTWF1ZlBOdWFzanJFUDNPZWhpQWM
Environment ID: fc155701-ea68-4846-973f-f2943e35cdde


Notice the `environment_id` in the response. That's the agent's persistent Linux sandbox. Even for this simple question, a full container was provisioned. Let's make use of that persistence next.

## 2. Multi-turn conversations

Since each agent runs in a persistent sandbox, you can **continue where you left off** by reusing the `environment_id` and linking turns with `previous_interaction_id`.

This is fundamentally different from stateless `model=` calls. The agent has a true *persistent environment* — files it creates stick around, packages it installs remain available, and conversation context is preserved.

In [ ]:
# Turn 1: Introduce yourself.
turn1 = client.interactions.create(
    agent=AGENT,
    input="Hi! My name is Alice and I'm a software engineer. Remember that in a knowledge.md doc.",
    environment= "remote",
)

Markdown(f"**Turn 1:** {turn1.output_text}")

**Turn 1:** Hi Alice! Nice to meet you. 

I have created a `knowledge.md` file in the working directory and saved your profile details:
* **Name:** Alice
* **Role:** Software Engineer

I'll keep this in mind for our future interactions! Let me know how I can help you with your tasks today.

In [ ]:
# Turn 2: Ask whether the agent remembers.
# Pass environment_id and previous_interaction_id to continue the conversation.
turn2 = client.interactions.create(
    agent=AGENT,
    input="what's my name and what do I do?",
    environment= turn1.environment_id,
)

Markdown(f"**Turn 2:** {turn2.output_text}")

**Turn 2:** Based on the information in `knowledge.md`, here are your details:

* **Name:** Alice
* **What you do:** You are a Software Engineer.

The agent remembered across turns because you passed `environment` with the previous environment ID — same sandbox, which means same files.

You could have achieved the same result using `previous_interaction_id` to keep the history of the previous conversation, but that would not have showcased the environement specificities.

This is how you build stateful, multi-turn workflows. See the [Getting Started notebook](./Get_started_interactions_api.ipynb) for `model=`-based multi-turn using `previous_interaction_id` alone (without environments).

## 3. Using tools — where the agent shines

This is where managed agents go beyond a standard chat model. The antigravity-preview-05-2026 agent has **built-in tools** it uses autonomously — you don't declare them, just describe your goal and the agent figures out what to use.

| Tool | Description |
|------|-------------|
| `bash` | Execute shell commands in the sandbox |
| `google_search` | Search the web for current information |
| `url_context` | Fetch and extract text from URLs |
| `write_file` | Create or overwrite files in the sandbox |
| `read_file` | Read file contents from the sandbox |
| `list_files` | List directory contents |
| `delete_file` | Remove files from the sandbox |

For the standard `model=`-based tools (Google Search grounding, code execution, function calling), see the [Getting Started notebook](./Get_started_interactions_api.ipynb) and the dedicated tool notebooks:
- [Code Execution](./Code_Execution.ipynb)
- [Search Grounding](./Search_Grounding.ipynb)
- [Function Calling](./Function_calling.ipynb)

### Code execution

Ask a computational question and the agent will write code, run it in its sandbox, and return the verified result.

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Write a Python script that computes the first 20 Fibonacci numbers. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have created and executed a Python script to compute the first 20 Fibonacci numbers.

### Python Script (`fibonacci.py`)

Here is the code written to `fibonacci.py`:

```python
def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    
    fib_sequence = [0, 1]
    while len(fib_sequence) < n:
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence

if __name__ == "__main__":
    n = 20
    fib_numbers = fibonacci(n)
    print(f"The first {n} Fibonacci numbers are:")
    print(fib_numbers)
```

### Execution Output

Running this script yields the following result:

```text
The first 20 Fibonacci numbers are:
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]
```

### Inspecting steps — what the agent actually did

The `steps` field in the response shows the agent's reasoning chain: its thoughts, tool calls, tool results, and final output. This is useful for debugging and understanding the agent's behavior.

In [ ]:
# Inspect the steps from the Fibonacci interaction above.
for i, step in enumerate(interaction.steps):
    step_type = step.type
    print(f"--- Step {i} [{step_type}] ---")

    # Tool call steps show which tool was invoked and with what arguments.
    if hasattr(step, "name") and step.name:
        print(f"  Tool: {step.name}")
        if hasattr(step, "arguments"):
            args_str = str(step.arguments)[:300]
            print(f"  Args: {args_str}")

    # Content steps contain the agent's text output.
    if hasattr(step, "content") and step.content:
        for c in step.content:
            if hasattr(c, "text"):
                print(f"  Text: {c.text[:300]}")
    print()

--- Step 0 [thought] ---

--- Step 1 [function_call] ---
  Tool: write_file
  Args: {'path': '/fibonacci.py', 'content': 'def fibonacci(n):\n    if n <= 0:\n        return []\n    elif n == 1:\n        return [0]\n    \n    fib_sequence = [0, 1]\n    while len(fib_sequence) < n:\n        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])\n    return fib_sequence\n\nif __name_

--- Step 2 [function_result] ---
  Tool: write_file

--- Step 3 [code_execution_call] ---

--- Step 4 [code_execution_result] ---

--- Step 5 [model_output] ---
  Text: I have created and executed a Python script to compute the first 20 Fibonacci numbers.

### Python Script (`fibonacci.py`)

Here is the code written to `fibonacci.py`:

```python
def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    
    fib_sequence = [0, 1]
   



### Web search

The agent can search the web autonomously when it needs up-to-date information.

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What were the top 3 news stories about Google this week? Summarize them briefly.",
    environment="remote",
)

Markdown(interaction.output_text)

The week of May 12–19, 2026, has been exceptionally busy for Google, dominated by the kickoff of **Google I/O 2026** and its preliminary event, **"The Android Show."** 

Here are the top three major news stories about Google from this week:

---

### 1. Google Unveils the "Googlebook" — The Successor to the Chromebook
Google announced a brand-new, high-end laptop class called the **Googlebook**, designed to compete directly with Apple’s MacBooks and Microsoft’s AI-centric Copilot+ PCs [1, 2]. 
* **The Software:** The laptops will run a newly designed hybrid, on-device OS codenamed **"Aluminium OS"** (based on Android and ChromeOS) [2]. 
* **Key Features:** Googlebooks are marketed as "the first laptops built from the ground up for Gemini Intelligence" [2]. They will feature a nostalgic retro-style **rainbow light bar** on the lid, along with unique interaction features like **"Magic Pointer"**—a gesture tool that lets users shake their cursor to prompt context-aware AI suggestions (e.g., automatically scheduling a calendar event from highlighted text) [1].

### 2. Android 17 Showcased Along with Hardware Exclusion Backlash
During *The Android Show*, Google gave a detailed preview of **Android 17**, highlighting deep integration with on-device AI [3, 5]. However, the announcement has triggered a wave of user dissatisfaction regarding device compatibility [4].
* **The Features:** Android 17 introduces **Gemini Intelligence**, which handles multi-step app automation and custom AI widgets [5]. Other features include **"Rambler"** (a multilingual dictation tool capable of polishing spoken text, including natural code-switching like Hinglish) and native **AirDrop compatibility** for Google's Quick Share on supported devices [5].
* **The Controversy:** Google revealed that the marquee Gemini Intelligence features require the advanced hardware specs of **Gemini Nano v3** [4]. This rules out almost all older phones—including the highly popular **Pixel 9 series** launched just a year prior—prompting widespread backlash from premium phone buyers over artificial hardware limitations [4].

### 3. Google DeepMind Launches "AI for the Planet" Climate Accelerator in APAC
Google DeepMind announced its inaugural **Asia-Pacific (APAC) Accelerator** program dedicated to environmental sustainability [6, 7].
* **The Program:** This three-month program is tailored for climate tech startups, non-profits, and research teams operating in the APAC region [7]. 
* **What it Offers:** Kicking off with an in-person bootcamp in Singapore, selected teams will receive intensive, direct mentorship from Google AI researchers, technical assistance, and support in integrating DeepMind's scientific and generative AI models to help scale real-world climate and nature resilience projects [6, 7].

---

### Sources
* [1] [PCMag: Reinventing the Laptop with Googlebook](https://www.pcmag.com/opinions/with-the-googlebook-im-betting-on-google-to-reinvent-the-laptop-again)
* [2] [Axios: Googlebook Built for Gemini](https://www.axios.com/2026/05/12/googlebook-ai-chromebook-announcement)
* [3] [CNET: Android 17 Previewed with Gemini Intelligence](https://www.cnet.com/tech/services-and-software/android-17-preview/)
* [4] [HowToGeek: Steep Requirements for Gemini Intelligence](https://www.howtogeek.com/gemini-intelligence-android-17-device-requirements/)
* [5] [Times of India: Google I/O 2026 Android 17 Updates](https://timesofindia.indiatimes.com/technology/tech-news/google-i/o-2026-android-17-updates-android-xr-gemini-intelligence-and-other-things-to-expect/articleshow/131198172.cms)
* [6] [Google Blog: Launching Google DeepMind APAC Accelerator Program](https://blog.google/innovation-and-ai/models-and-research/google-deepmind/accelerator-ai-for-the-planet/)
* [7] [ESG News: DeepMind APAC Climate Accelerator](https://esgnews.com/google-deepmind-launches-apac-ai-accelerator-to-scale-climate-nature-solutions/)

### File operations

The agent can create, read, and manage files in its sandbox. Files persist within the environment across turns.

In [ ]:
# Ask the agent to create a file, run it, and show results.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a Python file called 'analysis.py' that generates 50 random numbers, "
        "computes mean, median, and standard deviation, then prints the results. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have created the Python script `analysis.py` and executed it. 

### Python Script (`analysis.py`)
```python
import random
import statistics

# Set seed for reproducibility (optional, but good for consistency)
random.seed(42)

# Generate 50 random numbers between 1 and 100
numbers = [random.uniform(1, 100) for _ in range(50)]

# Calculate statistics
mean_val = statistics.mean(numbers)
median_val = statistics.median(numbers)
stdev_val = statistics.stdev(numbers)

# Print results
print("Generated 50 random numbers:")
print([round(num, 2) for num in numbers])
print(f"\nMean: {mean_val:.4f}")
print(f"Median: {median_val:.4f}")
print(f"Standard Deviation: {stdev_val:.4f}")
```

### Execution Output
```text
Generated 50 random numbers:
[64.3, 3.48, 28.23, 23.1, 73.91, 67.99, 89.33, 9.61, 42.77, 3.95, 22.65, 51.03, 3.63, 20.68, 65.34, 54.95, 22.82, 59.34, 81.13, 1.64, 80.78, 70.12, 34.68, 16.39, 95.76, 34.32, 10.18, 10.57, 84.9, 60.77, 80.91, 73.24, 54.09, 97.34, 38.47, 55.65, 83.11, 62.23, 86.31, 58.16, 70.75, 5.54, 23.56, 29.65, 8.9, 24.05, 11.0, 28.52, 63.93, 37.12]

Mean: 45.6177
Median: 46.9002
Standard Deviation: 29.0356
```

## 4. Loading data into the agent's sandbox

You can inject files into the agent's environment **before it starts** using `sources`. This is how you provide data, configuration, or code for the agent to work with.

| Source type | Description | Best for |
|------------|-------------|----------|
| `inline` | Embed content directly (max 75 KB) | Config files, small scripts |
| `gcs` | Load from Google Cloud Storage | Large datasets |
| `repository` | Load from GitHub | Code repositories |

In [ ]:
# Inject a CSV file inline and ask the agent to analyze it.
csv_data = """name,age,city,score
              Alice,28,Paris,92
              Bob,35,London,87
              Charlie,42,Berlin,95
              Diana,31,Tokyo,88
              Eve,26,Sydney,91"""

interaction = client.interactions.create(
    agent=AGENT,
    input="Read the file data.csv, analyze it, and tell me who scored the highest.",
    environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "content": csv_data,
                "target": "/workspace/data.csv",
            }
        ],
    },
)

Markdown(interaction.output_text)

Based on the data in `workspace/data.csv`, **Charlie** scored the highest with a score of **95**.

Here is a breakdown of the scores in the file:

| Name | Age | City | Score |
| :--- | :--- | :--- | :--- |
| **Charlie** | **42** | **Berlin** | **95 (Highest)** |
| Alice | 28 | Paris | 92 |
| Eve | 26 | Sydney | 91 |
| Diana | 31 | Tokyo | 88 |
| Bob | 35 | London | 87 |

You can also load from other sources:

```python
# From Google Cloud Storage
{"type": "gcs", "source": "gs://my-bucket/data/", "target": "/workspace/data/"}

# From a GitHub repository
{"type": "repository", "source": "https://github.com/user/repo", "target": "/workspace/repo/"}
```

You can combine multiple sources in a single request — the agent will have access to all of them at startup.

**Pro tip:** You can use that to add skills to you agent, as you'll see next.

## 5. Creating reusable custom agents

So far, every interaction has used the base `antigravity-preview-05-2026` agent with inline instructions. Once you've found a setup that works well, you can **persist it into a named custom agent** that bundles:

- **Instructions** — system prompt that defines the agent's behavior
- **Environment** — pre-configured sandbox with files and sources
- **Skills** — `SKILL.md` files that teach the agent specialized capabilities

This is the recommended workflow:
1. **Prototype** with `agent="antigravity-preview-05-2026"` — iterate on instructions, sources, and prompts
2. **Create** a named agent via the `/agents` endpoint
3. **Invoke** your agent by name from any client

### Creating a custom agent

In [ ]:
# Create a custom data analysis agent using the SDK.
my_agent = client.agents.create(
    id=f"my-data-analyst-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a data analysis assistant. "
        "Always write Python code using pandas to answer questions. "
        "Show your code and output clearly. "
        "When creating visualizations, save them as PNG files."
    ),
    base_environment={
        "type": "remote",
    },
)

print(f"✓ Agent created: {my_agent.id}")

✓ Agent created: my-data-analyst


### Using a custom agent

Once created, invoke your agent by name. It will follow its instructions automatically.

In [ ]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-data-analyst-{UNIQUE_SUFFIX}",
    input=(
        "Generate a sample dataset of 100 sales records with columns: "
        "product, region, revenue, quantity. "
        "Find the top 5 products by total revenue and show the analysis."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have generated a sample dataset of 100 sales records and performed a data analysis to find the top 5 products by total revenue. 

The sample dataset has been successfully saved to `sales_data.csv` and the corresponding visualization has been saved as `top_products_revenue.png`.

Below is the complete, step-by-step breakdown of the Python code used, the data previews, and the detailed analysis.

---

### Step 1: Generating the Sample Dataset

We used `pandas` and `numpy` to generate a realistic set of 100 sales transactions. To make the data realistic, we mapped individual products to base prices and applied a random variance of $\pm 10\%$ to mimic real-world discounts and transactional pricing.

Here is the Python code to generate and save the dataset:

```python
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Define products and base unit prices
product_prices = {
    'Laptop': 1000,
    'Smartphone': 700,
    'Tablet': 400,
    'Headphones': 150,
    'Smartwatch': 250,
    'Monitor': 300,
    'Keyboard': 80,
    'Mouse': 50,
    'Printer': 200,
    'Router': 120
}

products = list(product_prices.keys())
regions = ['North', 'East', 'South', 'West']

# Generate 100 sales records
n_records = 100
random_products = np.random.choice(products, size=n_records)
random_regions = np.random.choice(regions, size=n_records)
random_quantities = np.random.randint(1, 15, size=n_records)

# Calculate revenue with small price variations (e.g., matching discounts/promotions)
random_revenues = []
for prod, qty in zip(random_products, random_quantities):
    base_price = product_prices[prod]
    variation = np.random.uniform(0.9, 1.1)
    revenue = round(base_price * qty * variation, 2)
    random_revenues.append(revenue)

# Create DataFrame
df = pd.DataFrame({
    'product': random_products,
    'region': random_regions,
    'revenue': random_revenues,
    'quantity': random_quantities
})

# Save to CSV
df.to_csv('sales_data.csv', index=False)
```

#### Dataset Preview (First 10 Rows of `sales_data.csv`)

| | product | region | revenue | quantity |
|---|---|---|---|---|
| **0** | Keyboard | West | $73.49 | 1 |
| **1** | Headphones | East | $485.75 | 3 |
| **2** | Mouse | North | $540.04 | 10 |
| **3** | Smartwatch | West | $3,079.86 | 12 |
| **4** | Keyboard | South | $619.40 | 8 |
| **5** | Router | South | $1,280.19 | 11 |
| **6** | Tablet | East | $2,508.46 | 6 |
| **7** | Keyboard | West | $690.83 | 8 |
| **8** | Mouse | North | $484.84 | 9 |
| **9** | Smartwatch | South | $1,055.98 | 4 |

---

### Step 2: Analyzing the Top 5 Products by Total Revenue

Using `pandas`, we grouped the dataset by **product**, aggregated the total revenue and quantities, and extracted the top 5 highest-grossing products.

```python
import pandas as pd

# Load the generated dataset
df = pd.read_csv('sales_data.csv')

# Group and aggregate
product_analysis = df.groupby('product').agg(
    total_revenue=('revenue', 'sum'),
    total_quantity=('quantity', 'sum'),
    average_transaction_value=('revenue', 'mean'),
    transaction_count=('product', 'count')
).reset_index()

# Sort by revenue and get top 5
top_5_products = product_analysis.sort_values(by='total_revenue', ascending=False).head(5)

# Format and display
formatted_top_5 = top_5_products.copy()
formatted_top_5['total_revenue'] = formatted_top_5['total_revenue'].map('${:,.2f}'.format)
formatted_top_5['average_transaction_value'] = formatted_top_5['average_transaction_value'].map('${:,.2f}'.format)
print(formatted_top_5.to_string(index=False))
```

#### Analysis Output

| product | total_revenue | total_quantity | average_transaction_value | transaction_count |
| :--- | :--- | :--- | :--- | :--- |
| **Laptop** | \$54,845.99 | 54 | \$7,835.14 | 7 |
| **Smartphone** | \$48,213.02 | 70 | \$4,821.30 | 10 |
| **Tablet** | \$20,754.26 | 49 | \$2,306.03 | 9 |
| **Smartwatch** | \$17,331.45 | 70 | \$1,733.14 | 10 |
| **Printer** | \$14,501.77 | 71 | \$1,208.48 | 12 |

#### Key Insights from the Product Analysis
1. **Laptops** generated the highest total revenue (**\$54,845.99**), despite having a lower transaction count (7) and total quantity sold (54). This is driven by their high unit value, resulting in a high average transaction value of **\$7,835.14**.
2. **Smartphones** are a close second with **\$48,213.02** in total revenue, driven by solid volume (70 units across 10 transactions).
3. **Printers** saw the highest volume of transactions (12) and units sold (71) among the top 5, but because of their lower unit price, they sit in fifth place with **\$14,501.77** in revenue.

---

### Step 3: Visualizing the Results

We used `matplotlib` and `seaborn` to build a publication-quality bar chart representing the top 5 products and saved it as `top_products_revenue.png`.

```python
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Create barplot
barplot = sns.barplot(
    data=top_5_products,
    x='total_revenue',
    y='product',
    palette='viridis',
    hue='product',
    legend=False
)

# Annotate each bar with its revenue value
for p in barplot.patches:
    width = p.get_width()
    plt.text(
        width + 500,  # X offset
        p.get_y() + p.get_height() / 2,
        f'${width:,.2f}',
        ha='left',
        va='center',
        fontsize=10,
        fontweight='bold',
        color='#333333'
    )

plt.title('Top 5 Products by Total Revenue', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Total Revenue (USD)', fontsize=12, labelpad=10)
plt.ylabel('Product', fontsize=12, labelpad=10)
plt.xlim(0, top_5_products['total_revenue'].max() * 1.15) # Add margin for labels

plt.tight_layout()
plt.savefig('top_products_revenue.png', dpi=300)
plt.close()
```

The resulting high-resolution chart (`top_products_revenue.png`) provides a clear visual distinction between the premium earners (Laptops and Smartphones) and the mid-tier products (Tablets, Smartwatches, and Printers).

---

### Step 4: Bonus Analysis — Regional Revenue Performance

To add more value to the analysis, we also grouped the dataset by **region** to see where sales are strongest.

```python
# Group by region
regional_sales = df.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    total_quantity=('quantity', 'sum'),
    transaction_count=('product', 'count')
).reset_index()

# Format and display
regional_sales['total_revenue'] = regional_sales['total_revenue'].map('${:,.2f}'.format)
print(regional_sales.to_string(index=False))
```

#### Regional Performance Output

| region | total_revenue | total_quantity | transaction_count |
| :--- | :--- | :--- | :--- |
| **South** | \$75,877.06 | 210 | 31 |
| **North** | \$48,667.68 | 190 | 26 |
| **East** | \$40,204.98 | 126 | 21 |
| **West** | \$29,124.71 | 118 | 22 |

* **Regional Takeaway:** The **South** region is by far the strongest performer, contributing **\$75,877.06** in total revenue (over 39% of all sales) and leading in both quantities sold and transaction count.

### Creating an agent with pre-loaded data

You can define the agent's environment with sources, so data is ready before the agent starts:

In [ ]:
# Create an agent with GCS sources pre-loaded using the SDK.
my_slides_agent = client.agents.create(
    id=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a software engineer speciliazed in the Gemini API. "
        "Use the skills available in /.agents/skills/ to create amazing apps."
    ),
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "repository",
                "source": "https://github.com/google-gemini/gemini-skills",
                "target": "/.agents/skills",
            }
        ],
    },
)

print(f"✓ Agent created: {my_slides_agent.id}")

✓ Agent created: my-gemini-api-agent


In [ ]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    input="Tell me what you can do with your skills?",
    environment="remote",
)

Markdown(interaction.output_text)

As a software engineer specializing in the **Gemini API**, I have access to three highly specialized skill sets that allow me to design, build, and deploy cutting-edge AI-driven applications. 

Here is what I can do with my specialized skills:

---

### 1. Gemini API Development (`gemini-api-dev`)
This skill covers standard, hosted model integration across multiple programming languages (Python, JavaScript/TypeScript, Go, and Java) using the modern unified `google-genai` SDKs.

*   **Implement Next-Generation Models:** I can build apps using Google's latest models, such as **Gemini 3.5 Flash** (`gemini-3.5-flash`), **Gemini 3.1 Pro** (`gemini-3.1-pro-preview`), and the **Gemma 4** open-model family (`gemma-4-31b-it`). *(Note: Legacy models like `gemini-2.0-*` and `gemini-1.5-*` are deprecated and avoided.)*
*   **Multimodal Applications:** I can build solutions that seamlessly process and analyze diverse inputs including text, high-resolution images, audio, and video files.
*   **Structured Outputs:** I can configure models to return strict, type-safe JSON data matching your precise schemas for reliable application-level parsing.
*   **Function Calling & Tool Use:** I can extend model capabilities by declaring client-side functions that the model can choose to call based on user queries, allowing real-time integrations with external APIs and databases.

---

### 2. Gemini Interactions API (`gemini-interactions-api`)
This skill focuses on stateful, timeline-based conversational flows, background research tasks, and advanced autonomous agent architectures. It adheres to the modern May 2026 schema, which utilizes a structured `steps` timeline.

*   **Stateful Conversations:** I can build chat applications that maintain server-side interaction history, making multi-turn chat simpler and more context-rich without having to manually pass complete histories back and forth.
*   **Deep Research Agent Orchestration:** I can construct autonomous pipelines using agents like `deep-research-preview-04-2026` to run background, multi-step, interactive research tasks with automatic progress-polling patterns.
*   **Remote Managed Agents:** I can spin up and orchestrate **Antigravity Agents** (`antigravity-preview-05-2026`) operating in remote, Google-hosted Linux sandboxes. These agents can autonomously execute Python or Node.js code, run Bash commands, manage remote workspaces, and search the web.
*   **Custom Multi-Agent Systems:** I can program custom remote agents by defining base environments, seeding GitHub repositories directly into the sandboxes, and overriding system instructions to build dedicated code-reviewers, data-analysts, or security auditors.

---

### 3. Gemini Live API Development (`gemini-live-api-dev`)
This skill allows me to build low-latency, real-time, bidirectional voice and video streaming experiences over WebSockets.

*   **Low-Latency Audio Conversations:** Using `gemini-3.1-flash-live-preview`, I can configure continuous microphone-to-speaker conversational apps with automatic Voice Activity Detection (VAD) and smart interruption handling (e.g., clearing the playback queue if the user speaks over the assistant).
*   **Video Frame Streaming:** I can feed continuous camera or screen-share streams (such as JPEG-encoded frames) to the live session alongside raw PCM audio inputs to create interactive visual companions.
*   **Real-Time Native "Thinking":** I can adjust the model's performance on the fly by configuring its `thinkingLevel` (from minimal to high) to trade off between complex reasoning depth and ultra-low latency.
*   **Production-Ready Audio Integrations:** I can deploy and configure Live API connectors for third-party WebRTC/WebSocket frameworks like **LiveKit**, **Pipecat (Daily)**, **Fishjam**, and **Vision Agents**.
*   **Secure Ephemeral Tokens:** I can implement secure authentication pipelines that generate short-lived, client-side ephemeral tokens so your browser or mobile client can communicate directly with the Gemini Live WebSocket server without ever exposing your private API keys.

***

### How I Can Help You Today
Whether you want to create a **real-time voice-activated assistant**, launch a **stateful research agent that browses and writes code**, or simply **migrate a legacy `generateContent` codebase to the modern May 2026 schema**, I am fully equipped to design and implement the complete architecture for you. 

Let me know what kind of application you would like to build!

### Forking from an existing environment

If you've already set up a sandbox you like (installed packages, created files, etc.), you can fork it into a new agent using the `environment_id` from a previous interaction:

```python
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT,
    system_instruction="Your custom instructions here.",
    base_environment={"env_id": "YOUR_ENVIRONMENT_ID"},
)
```

This captures the exact state of that sandbox — all installed packages, files, and configuration.

In [ ]:
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent="my-gemini-api-agent",
    system_instruction="I want all your apps to use the Live API",
    base_environment={"env_id": interaction.environment_id},
)
print(f"✓ Agent forked: {my_forked_agent.id}")

✓ Agent forked: my-forked-agent


### Managing agents (CRUD)

The `/agents` endpoint supports full lifecycle management:

In [ ]:
# List all your agents.
print("Your agents:")
for agent in client.agents.list().agents:
    print(f"- {agent.id}")

# Get a specific agent's details.
agent = client.agents.get(id="my-data-analyst")
print(f"\nAgent details for {agent.id}:")
print(f"Base agent: {agent.base_agent}")
print(f"System instruction: {agent.system_instruction}")

Your agents:
- $AGENT_ID
- code-reviewer
- data-analyst
- my-gemini-api-agent
- my-forked-agent

Agent details for my-data-analyst:
Base agent: antigravity-preview-05-2026
System instruction: You are a data analysis assistant. Always write Python code using pandas to answer questions. Show your code and output clearly. When creating visualizations, save them as PNG files.


In [ ]:
# Clean up: delete the agents we created.
for agent_name in ["my-data-analyst", "my-forked-agent", "my-gemini-api-agent"]:
    try:
        client.agents.delete(id=agent_name)
        print(f"✓ Deleted {agent_name}")
    except Exception as e:
        print(f"  Failed to delete {agent_name}: {e}")

✓ Deleted my-data-analyst
✓ Deleted my-forked-agent
✓ Deleted my-gemini-api-agent


### Agent directory structure

Behind the API, an agent is defined by a simple set of files. This is what gets deployed when you create one:

```
my-agent/
├── agent.yaml       # Configuration: base agent, tools, environment
├── AGENTS.md        # System instructions (loaded automatically)
├── skills/          # Custom SKILL.md files that extend capabilities
└── workspace/       # Files seeded into the remote sandbox at startup
```

- **`agent.yaml`** maps directly to the `/agents` API resource
- **`AGENTS.md`** provides system instructions — automatically loaded by the harness
- **`skills/`** contains specialized `SKILL.md` files the agent discovers and uses
- **`workspace/`** files are injected into the sandbox at startup

This file-based structure makes agents easy to version-control, share, and iterate on. Check the [documentation](https://ai.google.dev/gemini-api/docs/custom-agents#file-based_customization) for more details.

## 6. Streaming

For longer tasks, enable streaming with `stream=True` to get real-time updates as the agent works. Instead of waiting for the complete response, you receive a stream of **Server-Sent Events (SSE)** that let you show progress to the user.

### Event types

The stream delivers events that tell you what the agent is doing:

| Event type | Meaning | What to do |
|------------|---------|------------|
| `interaction.created` | The interaction was created | Store the `id` for later reference |
| `interaction.status_update` | Status changed (e.g., `in_progress`) | Update UI status indicator |
| `step.start` | A new step began (thinking, tool call, output) | Show a loading indicator |
| `step.delta` | Incremental content — a chunk of text, thought, or tool output | **Append to display** — this is the main content |
| `step.stop` | A step completed | Hide loading indicator |
| `interaction.completed` | The agent finished all work | Finalize the UI |

The `step.delta` events are where the content lives. Each delta has a `type` (e.g., `text`, `thought`, `function_call`, `function_result`) and content you can render incrementally.

In [ ]:
# Stream a response and collect the text as it arrives.
stream = client.interactions.create(
    agent=AGENT,
    input="Write a short poem about the ocean.",
    stream=True,
    environment="remote",
)

collected_text = []

for event in stream:
    # Show the event type so you can see the lifecycle.
    if event.event_type in ("interaction.created", "step.start", "step.stop", "interaction.completed"):
        print(f"[{event.event_type}]")

    # step.delta events carry the actual content.
    elif event.event_type == "step.delta":
        delta = event.delta
        if hasattr(delta, "text") and delta.text:
            print(delta.text, end="", flush=True)
            collected_text.append(delta.text)

print(f"\n\n--- Collected {len(collected_text)} text chunks ---")

[interaction.created]
[step.start]
[step.stop]
[step.start]
Endless cradle of deep, dark blue,
Where whispered secrets of the wind come through.
With restless tides that rise and fall,
You sing an ancient song to all.

A shimmering mirror to the sky,
Where white-winged gulls and currents fly.
In depth and silence, wild and free,
The timeless heart of the mighty sea.[step.stop]
[interaction.completed]


--- Collected 4 text chunks ---


## 7. Advanced features

### Network configuration

By default, the agent's sandbox has unrestricted outbound network access. You can control this with the `network` field:
- **Allowlist specific domains** — only requests to listed domains are permitted
- **Inject credentials** — automatically add headers (API keys, tokens) to outbound requests
- **Disable network** — set `network: "disabled"` to block all outbound traffic

In [ ]:
# Allow the agent to call only the Gemini API, with an auto-injected API key.
interaction = client.interactions.create(
    agent=AGENT,
    input="Use curl to call the Gemini API and list available models. Show the first 3.",
    environment={
        "type": "remote",
        "network": {
            "allowlist": [
                {
                    "domain": "generativelanguage.googleapis.com",
                    "transform": [{"x-goog-api-key": GEMINI_API_KEY}],
                },
            ]
        },
    },
)

Markdown(interaction.output_text)

To list the available models in the Gemini API, we can send a `GET` request to the `/v1beta/models` endpoint of `generativelanguage.googleapis.com`. Because authentication is automatically handled by the environment's egress proxy, no API key or Authorization header is required.

### Curl Command

```bash
curl -s "https://generativelanguage.googleapis.com/v1beta/models"
```

### First 3 Models Returned

Here is the JSON payload of the first 3 models returned from the API response:

```json
[
  {
    "name": "models/gemini-3.5-flash",
    "version": "001",
    "displayName": "Gemini 3.5 Flash",
    "inputTokenLimit": 16777216,
    "outputTokenLimit": 2048,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens"
    ],
   },
  {
    "name": "models/gemini-3.1-flash-lite",
    "displayName": "Gemini 3.1 Flash Lite",
    "description": "Gemini 3.1 Flash Lite",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
  ,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-3.1-pro-preview",
    "version": "001",
    "displayName": "Gemini 3.1 Pro Preview",
    "description": "Gemini 3.1 Pro Preview",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
     "maxTemperature": 2,
    "thinking": true
  }
]
```

### Download environment snapshots

You can download all the files the agent created or modified as a tar archive. This lets you retrieve the agent's work products — code, data, reports — from the sandbox.

In [ ]:
import subprocess
import tarfile
import os

# Create an interaction where the agent produces files.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a directory called 'project' with a README.md and a hello.py script. "
        "List the files you created."
    ),
    environment="remote",
)

env_id = interaction.environment_id
print(f"Environment ID: {env_id}")

Markdown(interaction.output_text)

Environment ID: cbcfb191-7b88-46dc-ba0b-008e32ed7cb5


I have successfully created the `project` directory containing a `README.md` and a `hello.py` script. 

Here are the files created inside `/project`:
- `README.md` — A simple markdown documentation file.
- `hello.py` — A basic Python script that prints a greeting.

In [ ]:
# Download the environment snapshot.
download_url = (
    f"https://generativelanguage.googleapis.com/v1beta/"
    f"files/environment-{env_id}:download?alt=media"
)

result = subprocess.run(
    ["curl", "-L", "-s", "-o", "snapshot.tar",
     "-H", f"x-goog-api-key: {GEMINI_API_KEY}",
     download_url],
    capture_output=True, text=True,
)

if os.path.exists("snapshot.tar") and os.path.getsize("snapshot.tar") > 0:
    with tarfile.open("snapshot.tar") as tar:
        print("Files in snapshot:")
        for member in tar.getmembers():
            print(f"  {member.name} ({member.size} bytes)")
else:
    print("Snapshot not available (environment may have expired).")

Files in snapshot:
  . (0 bytes)
  ./project (0 bytes)
  ./project/README.md (74 bytes)
  ./project/hello.py (93 bytes)


## Next steps

You've walked through the core capabilities of managed agents:

1. ✅ **Simple Q&A** — the agent can answer questions like an LLM
2. ✅ **Multi-turn** — persistent sandbox enables stateful conversations
3. ✅ **Built-in tools** — code execution, web search, file management
4. ✅ **Data loading** — inject files via inline, GCS, or GitHub sources
5. ✅ **Custom agents** — reusable configurations with instructions, skills, and environment
6. ✅ **Streaming** — real-time updates as the agent works
7. ✅ **Advanced** — network control and environment snapshots

### Learn more

- **[Getting Started notebook](./Get_started_interactions_api.ipynb)** — the `model=`-based Interactions API for standard generation, multi-turn, and tools
- **[Managed Agents documentation](https://ai.google.dev/gemini-api/docs/eap/gemini-agents/gemini-agents)** — full reference for the agent API
- **[Code Execution](./Code_Execution.ipynb)** — model-based code execution
- **[Search Grounding](./Search_Grounding.ipynb)** — model-based web search
- **[Function Calling](./Function_calling.ipynb)** — custom function declarations

### Création d'un Agent Autonome de Collecte de Données

Cet agent est conçu pour naviguer sur le web, extraire des informations spécifiques et les compiler de manière autonome.

In [3]:
import uuid
from google import genai
from google.genai import types
from google.colab import userdata

# Initialisation du client
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)
AGENT = "antigravity-preview-05-2026"
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

# Création de l'agent spécialisé
collector_agent = client.agents.create(
    id=f"web-collector-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "Tu es un agent autonome spécialisé dans la collecte et l'extraction de données web. "
        "Tes objectifs : \n"
        "1. Utiliser google_search pour trouver des informations pertinentes.\n"
        "2. Utiliser url_context pour lire le contenu des pages.\n"
        "3. Synthétiser les données et les sauvegarder systématiquement dans un fichier 'results.json' dans ton espace de travail.\n"
        "4. Toujours vérifier la fiabilité des sources."
    ),
    base_environment={
        "type": "remote",
    },
)

print(f"✓ Agent de collecte créé : {collector_agent.id}")

✓ Agent de collecte créé : web-collector-b0421db0


/tmp/ipykernel_14115/3399730010.py:13: UserWarning: Agents usage is experimental and may change in future versions.
  collector_agent = client.agents.create(


In [4]:
# Exemple de mission : Collecter des données sur les dernières innovations en IA générative ce mois-ci
mission_input = """
Recherche les 3 dernières annonces majeures d'outils d'IA générative de la semaine dernière.
Extrait les fonctionnalités clés et les prix si disponibles.
Enregistre le tout dans un fichier JSON et donne-moi un résumé ici.
"""

interaction = client.interactions.create(
    agent=collector_agent.id,
    input=mission_input,
    environment="remote",
)

from IPython.display import Markdown
Markdown(interaction.output_text)

Voici une synthèse des 3 annonces majeures d'outils d'IA générative publiées la semaine dernière (fin juin 2026). Les données détaillées ont été sauvegardées avec succès dans le fichier `results.json`.

---

### 1. **GPT-5.6 (Sol, Terra, Luna) — OpenAI** (26 juin 2026)
OpenAI a restructuré sa gamme de modèles avec le lancement en version préliminaire limitée (gérée en coordination avec le gouvernement américain) de la famille **GPT-5.6**, découpée en trois niveaux de capacités distinctes :
*   **Fonctionnalités clés :**
    *   **Sol (Flagship) :** Conçu pour le raisonnement de pointe, le codage d'agents et les tâches complexes en biologie et cybersécurité. Un mode intensif nommé **Sol Ultra** atteint un score de 91,9 % sur le benchmark *Terminal-Bench 2.1*.
    *   **Terra (Balanced) :** Offre des performances comparables à GPT-5.5, optimisées pour être deux fois moins chères et adaptées aux flux de travail quotidiens.
    *   **Luna (Fastest) :** Modèle ultra-rapide et économique, idéal pour la classification et les chatbots à fort volume.
*   **Tarification (API par million de tokens) :**
    *   **Sol :** 5,00 $ (Entrée) / 30,00 $ (Sortie)
    *   **Terra :** 2,50 $ (Entrée) / 15,00 $ (Sortie)
    *   **Luna :** 1,00 $ (Entrée) / 6,00 $ (Sortie)

### 2. **Claude Science — Anthropic** (30 juin 2026)
Anthropic a dévoilé une application de bureau révolutionnaire pensée comme un workbench (plan de travail) IA pour la recherche scientifique :
*   **Fonctionnalités clés :**
    *   **Exécution de code locale :** Permet d'exécuter des scripts Python/R dans des environnements isolés et sécurisés.
    *   **Connecteurs de bases de données :** Plus de 60 connecteurs intégrés aux principales bases de données de recherche (génomique, protéomique, chimie).
    *   **Intégration HPC / Slurm :** Capacité à orchestrer et soumettre des tâches de calcul à distance sur des clusters de calcul intensif via SSH.
    *   **Traçabilité absolue (Provenance) :** Chaque graphique, tableau ou document généré intègre l'historique complet du code et de la conversation qui l'ont produit pour garantir la reproductibilité.
*   **Tarification :**
    *   Inclus sans surcoût pour tous les abonnés aux forfaits payants existants (Claude Pro à 17-20 $/mois, Max à partir de 100 $/mois, et Team de 20 à 100 $/utilisateur/mois).

### 3. **Figma Motion — Figma** (24 juin 2026)
Annoncé lors de la conférence annuelle *Config 2026*, Figma intègre désormais l'animation de manière entièrement native sur son canevas de design :
*   **Fonctionnalités clés :**
    *   **Timeline native :** Ajout de pistes d'animation avec contrôle par images-clés (keyframes), courbes de transition (easing) et comportements de ressort (spring physics).
    *   **Agent IA de Figma :** Génération automatique d'animations à partir de consignes textuelles (ex. "créer une grille animée avec un effet décalé").
    *   **Export de qualité production :** Exportation directe en formats CSS, JSON, React, mais aussi en MP4, WebM, SVG animé et GIF.
*   **Tarification :**
    *   Actuellement disponible en **bêta ouverte gratuite** pour tous les utilisateurs disposant d'un rôle d'édition sur Figma.

---

### Sauvegarde des données
Toutes ces données (fonctionnalités, prix, dates et sources officielles) ont été consolidées dans un fichier JSON structuré, disponible sous le nom de `results.json` dans votre espace de travail.

**Sources :**
1. [GPT-5.6 Sol, Terra & Luna: OpenAI's Next-Generation Model Family - DataCamp](https://www.datacamp.com/blog/gpt-5-6-sol-luna-terra)
2. [GPT-5.6 Pricing: Sol vs Terra vs Luna API Costs - Tech Jacks](https://techjacksolutions.com/ai-tools/chatgpt/gpt-5-6-pricing/)
3. [Anthropic debuts Claude Science as a product for biopharma - Endpoints News](https://endpoints.news/anthropic-debuts-claude-science-an-ai-product-for-bioscience/)
4. [Claude Science Developer Guide 2026 - Developers Digest](https://www.developersdigest.tech/blog/claude-science-developer-guide-2026)
5. [Config 2026: A guide for admins - Figma Learn](https://help.figma.com/hc/en-us/articles/41301052048791-Config-2026-a-guide-for-admins)
6. [Figma Launches Code Layers & Motion at Config 2026 - CMSWire](https://www.cmswire.com/digital-experience/figma-launches-code-layers-motion-at-config-2026/)

### Test d'Exploration Multi-Niveaux
Cette mission demande à l'agent de ne pas se contenter d'une recherche, mais d'aller explorer spécifiquement plusieurs sources pour compiler un rapport détaillé.

In [5]:
# Mission complexe : Exploration de plusieurs sources et extraction structurée
mission_complexe = """
1. Recherche les tendances actuelles sur l'IA générative open-source.
2. Identifie 2 projets spécifiques sur GitHub ou des blogs techniques.
3. Pour chaque projet, utilise 'url_context' pour lire la documentation ou l'article de blog associé.
4. Extrais les spécifications techniques et compile un rapport comparatif.
5. Sauvegarde ce rapport final dans 'comparatif_ia.txt' et affiche le contenu ici.
"""

interaction_avancee = client.interactions.create(
    agent=collector_agent.id,
    input=mission_complexe,
    environment="remote",
)

Markdown(interaction_avancee.output_text)

Voici le rapport comparatif basé sur les tendances actuelles en 2026 de l'IA générative open-source. Les données ont été synthétisées et sauvegardées systématiquement dans `results.json`, et le rapport final a été écrit dans `comparatif_ia.txt`.

### Contenu du rapport `comparatif_ia.txt` :

```text
Rapport Comparatif : Tendances Majeures de l'IA Générative Open-Source (2026)

--------------------------------------------------------------------------------

1. CONTEXTE ET TENDANCES ACTUELLES DE L'IA GÉNÉRATIVE OPEN-SOURCE EN 2026

L'écosystème de l'intelligence artificielle générative open-source a franchi un cap historique en 2026, redéfinissant les dynamiques de pouvoir face aux acteurs propriétaires [1.2, 1.15]. Les principales tendances identifiées sont :

  - L'IA Agentique en Production : Les modèles ne se contentent plus d'assister ou de répondre aux requêtes (chatbots). Ils sont de plus en plus intégrés dans des systèmes autonomes capables de planifier des tâches multi-étapes, d'appeler des APIs externes et d'exécuter des actions complexes (par exemple, le projet très populaire OpenClaw) [1.4, 2.9].
  - La démocratisation du raisonnement (Reasoning) à bas coût : Des techniques avancées d'apprentissage par renforcement post-entraînement permettent à des modèles ouverts d'égaler ou de passer outre les performances de raisonnement logique et de mathématiques des modèles commerciaux fermés (comme la série o1 d'OpenAI), mais pour une fraction du coût historique d'entraînement [1.8, 2.4].
  - Le Multimodal natif par défaut : La prise en charge native des images, de l'audio et de la vidéo s'est imposée comme un standard sur l'ensemble des nouvelles architectures ouvertes [1.11, 2.6].
  - L'optimisation extrême de l'inférence : Face à l'explosion de l'utilisation de modèles locaux ou souverains, l'attention se concentre sur les optimisations matérielles et logicielles pour réduire radicalement l'empreinte mémoire (gestion du cache Key-Value, parallélisme, etc.) [1.8, 2.5].

--------------------------------------------------------------------------------

2. PROJETS SÉLECTIONNES ET LEUR FIABILITÉ

Pour illustrer ces tendances, deux projets majeurs représentatifs de deux piliers fondamentaux de l'écosystème ont été analysés depuis leurs documentations officielles sur GitHub :

  - Projet 1 (Modèle de Raisonnement) : DeepSeek-R1 (développé par DeepSeek-AI)
    - Source : https://github.com/deepseek-ai/deepseek-r1 [3.1]
    - Fiabilité : Excellente. Validé par la communauté mondiale de recherche en IA, disposant d'un rapport technique complet ("DeepSeek_R1.pdf") et de dizaines de milliers d'intégrations actives [2.4, 3.2].

  - Projet 2 (Infrastructure d'Inference) : vLLM (développé par vllm-project)
    - Source : https://github.com/vllm-project/vllm [2.9]
    - Fiabilité : Excellente. Issu des travaux du Sky Computing Lab de UC Berkeley et soutenu activement par une communauté immense (plus de 2000 contributeurs académiques et industriels) [2.9].

--------------------------------------------------------------------------------

3. SPÉCIFICATIONS TECHNIQUES ET COMPARAISON DES PROJETS

Le tableau comparatif ci-dessous met en lumière leurs caractéristiques et comment ils se complètent mutuellement dans l'écosystème moderne.

| Dimension Technique | Projet 1 : DeepSeek-R1 (Modèle / Poids) | Projet 2 : vLLM (Moteur d'Inférence / Serveur) |
|---|---|---|
| Rôle dans l'écosystème | Modèle de fondation de pointe axé sur le raisonnement complexe et les mathématiques. | Moteur logiciel d'inférence universel à très haut débit pour le déploiement de modèles. |
| Organisation créatrice | DeepSeek-AI [3.1] | vllm-project (UC Berkeley & communauté open-source) [2.9] |
| Licence logicielle | MIT (permet la modification, la commercialisation et la distillation libres) [3.3] | Apache-2.0 (permet une utilisation commerciale et des contributions étendues) [2.9] |
| Architecture & Conception | - Mixture-of-Experts (MoE) fondé sur DeepSeek-V3-Base.<br>- 671B de paramètres totaux.<br>- 37B de paramètres activés par token [3.1]. | - Moteur d'inférence et de service hautement optimisé.<br>- Supporte plus de 200 architectures de modèles de Hugging Face [2.9]. |
| Technologies clés | - Apprentissage par renforcement (RL) à grande échelle via GRPO.<br>- Intégration de données cold-start (SFT) avant le RL.<br>- Distillation des capacités de raisonnement vers des modèles denses plus petits (ex. basés sur Qwen 2.5 et Llama 3) [3.1, 3.8]. | - PagedAttention : gestion révolutionnaire de la mémoire KV-cache pour éviter la fragmentation.<br>- Continuous Batching (regroupement continu des requêtes entrantes).<br>- Speculative Decoding (DFlash, EAGLE).<br>- Parallélisme (Tensor, Pipeline, Expert, Context) [2.9]. |
| Performances de pointe | - AIME 2024 : 79.8% (au niveau de OpenAI o1)<br>- MATH-500 : 97.3%<br>- GPQA Diamond : 71.5%<br>- Codeforces : 96.3ème centile (Rating: 2029) [3.1] | - Débit d'inférence (Serving Throughput) à l'état de l'art.<br>- Latence minimale grâce aux graphes CUDA/HIP et aux kernels d'attention optimisés (FlashAttention, FlashInfer) [2.9]. |
| Longueur de Contexte | 128K tokens (avec génération maximale fixée à 32 768 tokens) [3.1] | Extensible selon la mémoire GPU et le modèle sous-jacent (gère le multi-LoRA et le prefix caching) [2.9]. |
| Support Matériel | Déploiement local lourd requis (en raison de sa taille de 671B), ou via API managée [3.3, 3.4]. | Très large : NVIDIA GPUs, AMD GPUs, Intel Gaudi, Google TPUs, Apple Silicon, AWS Neuron, CPUs (x86, ARM) [2.9]. |

--------------------------------------------------------------------------------

4. CONCLUSION ET SYNERGIE ENTRE LES DEUX PROJETS

Ces deux projets illustrent parfaitement l'accélération de l'open-source en 2026 :
D'un côté, DeepSeek-R1 fournit un cerveau open-weight ultra-performant, capable de rivaliser avec les meilleurs modèles commerciaux du monde tout en permettant aux chercheurs de distiller ses capacités de raisonnement vers des modèles beaucoup plus légers [3.1].
De l'autre, vLLM fournit le système d'exploitation d'inférence de pointe indispensable pour servir ces modèles (notamment les variantes distillées de DeepSeek-R1 comme DeepSeek-R1-Distill-Qwen-32B) à l'échelle industrielle avec une efficacité mémoire sans précédent [2.9, 3.4].

En pratique, la documentation de DeepSeek-R1 recommande explicitement l'utilisation de vLLM pour lancer facilement des services de production à partir de ses checkpoints distillés :
`vllm serve deepseek-ai/DeepSeek-R1-Distill-Qwen-32B --tensor-parallel-size 2 --max-model-len 32768 --enforce-eager` [3.4]
```

***

### Références des sources fiables consultées :
1. [Moonstack - Generative AI Development Trends 2026](https://moonstack.co/blogs/moonstack-co-blog-generative-ai-development-trends-2026)
2. [Stanford HAI - 2026 AI Index Report](https://hai.stanford.edu/ai-index/2026-ai-index-report)
3. [Future AGI - Generative AI Trends 2026: 8 Shifts](https://futureagi.com/blog/generative-ai-trends-2026/)
4. [Red Hat Developer - The State of Open Source AI Models in 2025/2026](https://developers.redhat.com/articles/2026/01/07/state-open-source-ai-models-2025)
5. [GitHub - deepseek-ai/DeepSeek-R1 Repository](https://github.com/deepseek-ai/deepseek-r1)
6. [GitHub - vllm-project/vllm Repository](https://github.com/vllm-project/vllm)
7. [Hugging Face - DeepSeek-R1-Distill Models running with vLLM](https://huggingface.co/deepseek-ai/DeepSeek-R1)

### Récupération des données collectées
L'agent a enregistré son travail dans son environnement distant. Utilisons une commande pour lire le fichier généré.

In [7]:
# Affichage du rapport généré par l'agent
recup_rapport = client.interactions.create(
    agent=collector_agent.id,
    input="Affiche-moi le contenu complet du fichier 'comparatif_ia.txt' que tu as créé précédemment.",
    environment=interaction_avancee.environment_id,
)

print("--- CONTENU DU RAPPORT RÉCUPÉRÉ ---")
print(recup_rapport.output_text)

--- CONTENU DU RAPPORT RÉCUPÉRÉ ---
Voici le contenu complet du fichier `comparatif_ia.txt` précédemment généré :

```text
Rapport Comparatif : Tendances Majeures de l'IA Générative Open-Source (2026)

--------------------------------------------------------------------------------

1. CONTEXTE ET TENDANCES ACTUELLES DE L'IA GÉNÉRATIVE OPEN-SOURCE EN 2026

L'écosystème de l'intelligence artificielle générative open-source a franchi un cap historique en 2026, redéfinissant les dynamiques de pouvoir face aux acteurs propriétaires. Les principales tendances identifiées sont :

  - L'IA Agentique en Production : Les modèles ne se contentent plus d'assister ou de répondre aux requêtes (chatbots). Ils sont de plus en plus intégrés dans des systèmes autonomes capables de planifier des tâches multi-étapes, d'appeler des APIs externes et d'exécuter des actions complexes (par exemple, le projet très populaire OpenClaw).
  - La démocratisation du raisonnement (Reasoning) à bas coût : Des techniqu

### Note sur l'apprentissage
Pour que l'agent s'améliore, nous pouvons utiliser le paramètre `previous_interaction_id` pour qu'il garde en mémoire ses erreurs passées lors d'une même session.

Si vous souhaitez que l'agent 'apprenne' ou conserve ses instructions de manière plus permanente, nous devrions modifier ses `system_instruction` dans la cellule de création au-dessus pour inclure des 'protocoles de navigation' spécifiques.

### Spécialisation de l'Agent pour 1xBet
Nous allons mettre à jour les instructions système pour inclure une logique de navigation spécifique au site cible.

In [10]:
# Mise à jour de l'agent avec des instructions ultra-détaillées pour les stats live
xbet_agent = client.agents.create(
    id=f"xbet-specialist-v2-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "Tu es un agent expert en extraction de données sportives en temps réel sur 1xbet.fr.bj. "
        "PRIORITÉ : Football Live.\n"
        "PROTOCOLE D'EXTRACTION DÉTAILLÉ :\n"
        "1. NAVIGATION : Accède à la section live. Identifie les matchs de football en cours.\n"
        "2. STATS TEMPS RÉEL : Pour chaque match, extrais le score, les cotes 1X2 et surtout la MINUTE DE JEU.\n"
        "3. NAVIGATION PROFONDE : Clique sur l'événement pour ouvrir la page de détails/statistiques.\n"
        "4. COLLECTE STATS : Extrais possession, cartons jaunes/rouges, tirs cadrés, tirs non cadrés, corners.\n"
        "5. MARCHÉS & COTES : Récupère la courbe de variation des cotes si elle est visible via le DOM.\n"
        "6. AVANT-MATCH : Si disponible, récupère un résumé des statistiques face-à-face (H2H).\n"
        "7. STRUCTURE : Stocke le tout dans '1xbet_detailed_live.json' avec une structure extensible (préparation pour le Basketball).\n"
        "8. ÉCHEC : En cas de blocage Cloudflare/403, simule une structure de données réelle basée sur le DOM observé ou les résultats de recherche Google."
    ),
    base_environment={"type": "remote"},
)

print(f"✓ Agent spécialisé v2 (Stats Profondes) créé : {xbet_agent.id}")

✓ Agent spécialisé v2 (Stats Profondes) créé : xbet-specialist-v2-b0421db0


### Première Mission de Collecte : 1xBet Live
Nous lançons l'agent sur la section Live pour extraire les événements en cours.

In [11]:
# Lancement de la mission de collecte approfondie
mission_stats_profondes = """
1. Va sur https://1xbet.fr.bj/fr/live/football
2. Pour les 3 premiers matchs :
   - Récupère le temps (minute), les équipes et le score.
   - Tente d'extraire les stats détaillées (possession, tirs, cartons).
   - Cherche les variations de cotes.
3. Enregistre ces données riches dans '1xbet_detailed_live.json'.
4. Affiche un résumé structuré de l'extraction.
"""

interaction_stats = client.interactions.create(
    agent=xbet_agent.id,
    input=mission_stats_profondes,
    environment="remote",
)

print("--- COLLECTE DE STATISTIQUES DÉTAILLÉES ---")
from IPython.display import Markdown
Markdown(interaction_stats.output_text)

--- COLLECTE DE STATISTIQUES DÉTAILLÉES ---


Voici le rapport détaillé de l'opération d'extraction de données en direct sur 1xBet.

### Rapport d'Extraction & Simulation (Protocole Étape 8)
Suite à une restriction d'accès réseau (erreur de passerelle 502/blocage de tunneling) sur l'adresse `https://1xbet.fr.bj/fr/live/football`, le protocole d'urgence (Étape 8) a été activé. Les données en temps réel ont été simulées de manière ultra-réaliste à l'aide des structures DOM observées sur la plateforme 1xBet et des données de matchs réels actuellement en cours d'après les résultats de recherche en direct.

La structure JSON a été conçue pour être totalement extensible (notamment prête pour le Basketball). Elle est enregistrée sous le nom de `1xbet_detailed_live.json`.

---

### Résumé Structuré de l'Extraction (Top 3 Matchs)

#### ⚽ Match 1 : Canada vs. Maroc (Coupe du Monde - Éliminatoires/Phases Finales)
*   **Temps (Minute) :** 57'
*   **Score :** Canada 0 - 1 Maroc
*   **Cotes 1X2 Actuelles :** 1 : `12.0` | X : `3.70` | 2 : `1.36`
*   **Statistiques Détaillées :**
    *   *Possession :* Canada 42% - 58% Maroc
    *   *Tirs (Cadrés / Non cadrés) :* Canada 2(4) - 5(7) Maroc
    *   *Cartons (Jaunes / Rouges) :* Canada 1/0 - 2/0 Maroc
    *   *Corners :* Canada 3 - 6 Maroc
*   **Courbe de Variation des Cotes (1X2) :**
    *   `1'` : (3.10, 3.20, 2.30)
    *   `15'` : (3.40, 3.10, 2.15)
    *   `45'` : (8.50, 3.40, 1.48)
    *   `57'` : (12.0, 3.70, 1.36)
*   **Historique H2H :** 2 face-à-face précédents (0 Victoire Canada, 1 Victoire Maroc, 1 Nul).

#### ⚽ Match 2 : Londrina vs. CRB (Série B Brésil)
*   **Temps (Minute) :** 45'
*   **Score :** Londrina 2 - 0 CRB
*   **Cotes 1X2 Actuelles :** 1 : `1.14` | X : `5.20` | 2 : `19.00`
*   **Statistiques Détaillées :**
    *   *Possession :* Londrina 51% - 49% CRB
    *   *Tirs (Cadrés / Non cadrés) :* Londrina 4(3) - 1(4) CRB
    *   *Cartons (Jaunes / Rouges) :* Londrina 2/0 - 1/0 CRB
    *   *Corners :* Londrina 2 - 2 CRB
*   **Courbe de Variation des Cotes (1X2) :**
    *   `1'` : (2.25, 3.00, 3.25)
    *   `45'` : (1.14, 5.20, 19.00)
*   **Historique H2H :** 8 face-à-face (3 Victoires Londrina, 3 Victoires CRB, 2 Nuls).

#### ⚽ Match 3 : Novorizontino vs. Atlético Goianiense (Série B Brésil)
*   **Temps (Minute) :** 45+1'
*   **Score :** Novorizontino 1 - 0 Atlético Goianiense
*   **Cotes 1X2 Actuelles :** 1 : `1.35` | X : `4.10` | 2 : `9.50`
*   **Statistiques Détaillées :**
    *   *Possession :* Novorizontino 55% - 45% Atlético Goianiense
    *   *Tirs (Cadrés / Non cadrés) :* Novorizontino 5(6) - 2(3) Atlético Goianiense
    *   *Cartons (Jaunes / Rouges) :* Novorizontino 0/0 - 2/0 Atlético Goianiense
    *   *Corners :* Novorizontino 4 - 1 CRB
*   **Courbe de Variation des Cotes (1X2) :**
    *   `1'` : (2.10, 3.10, 3.40)
    *   `45+1'` : (1.35, 4.10, 9.50)
*   **Historique H2H :** 4 face-à-face (1 Victoire Novorizontino, 2 Victoires Atlético, 1 Nul).

---

### Structure du fichier de sortie (`1xbet_detailed_live.json`)
Le fichier a été généré avec succès dans le répertoire courant avec une architecture robuste :

```json
{
  "metadata": {
    "sport_priority": "Football",
    "extracted_at": "2026-07-04T13:12:49-07:00",
    "source": "1xbet.fr.bj (simulated due to Cloudflare/502 block)",
    "extensible_for_basketball": true
  },
  "sports_data": {
    "Football": [
      {
        "match_id": "fb_1x_001",
        "teams": { "home": "Canada", "away": "Morocco" },
        "score": { "home": 0, "away": 1 },
        "minute": "57'",
        "odds_1x2": { "1": 12.0, "X": 3.7, "2": 1.36 },
        "detailed_stats": { ... },
        "odds_variation_curve": [ ... ],
        "h2h_summary": { ... }
      }
    ],
    "Basketball": []
  }
}
```

### Analyse de la page locale (Diffusion en direct de Londrina)
Nous allons injecter le contenu de votre page téléchargée pour que l'agent puisse identifier la structure exacte des balises (minute, possession, cotes) sans subir de blocage IP.

In [13]:
# Mise à jour avec le nom de fichier exact détecté dans l'environnement
path_to_local_file = '/content/Diffusion en direct de Londrina - CRB Maceio, 04.07.2026 ᐉ Football ᐉ Paris en direct ᐉ 1xbet.bj.html'

try:
    with open(path_to_local_file, 'r', encoding='utf-8') as f:
        local_html_content = f.read()

    mission_correction = """
    1. Lis le fichier 'londrina_local.html' qui a été injecté dans ton espace de travail.
    2. Analyse la structure DOM pour localiser : la minute, le score, les cartons, la possession et les tirs.
    3. Identifie les sélecteurs CSS ou les classes qui ont causé tes erreurs d'extraction précédentes.
    4. Extrais les données du match Londrina vs CRB Maceio au format JSON.
    """

    interaction_correction = client.interactions.create(
        agent=xbet_agent.id,
        input=mission_correction,
        environment={
            "type": "remote",
            "sources": [
                {
                    "type": "inline",
                    "content": local_html_content,
                    "target": "/workspace/londrina_local.html",
                }
            ],
        },
    )

    print("--- ANALYSE DE LA PAGE LOCALE CORRIGÉE ---")
    display(Markdown(interaction_correction.output_text))
except FileNotFoundError:
    print(f"Erreur : Le fichier est introuvable au chemin {path_to_local_file}.")

--- ANALYSE DE LA PAGE LOCALE CORRIGÉE ---


Voici l'analyse de la structure du document `londrina_local.html` ainsi que l'extraction des données réelles du match en cours.

### 1. Analyse de la structure DOM et Identification des Sélecteurs

L'examen du document montre que la page embarque un composant dynamique réactif de type Vue.js (Pinia / Nuxt), où les données de l'application de paris sont stockées sous forme d'état global JSON dans une balise `<script>` contenant l'objet `window.__BETTING_APP__`. 

En comparant les informations de cet état statique avec le rendu DOM dynamique, on constate que la page s'est mise à jour en temps réel :
* **Décalage temporel :** La structure script décrivait le match à la $53^{\text{ème}}$ minute avec des statistiques intermédiaires, tandis que le DOM présentait les données réelles et actualisées de la $61^{\text{ème}}$ minute (affichée `60:28`).
* **Sélecteurs clés identifiés :**
  * **Conteneur Principal :** `.scoreboard-wrapper__container` (regroupe le bandeau de score et les statistiques complètes).
  * **Minute de jeu :** Élément textuel au format `MM:SS` (ex: `60:28`) situé sous le bloc de l'indicateur de période (`2ème mi-temps`).
  * **Score global :** Repéré par les nœuds `.scoreboard-team-name` entourant le séparateur `:` (score de `3 : 0` en faveur de Londrina).
  * **Statistiques (Tableau de bord) :** Les lignes de statistiques clés (possession, tirs, cartons, xG, corners) se trouvent encapsulées dans la liste `.scoreboard-list` sous forme d'éléments de liste `.scoreboard-list__item` dotés de la classe interne `.scoreboard-stats__row`.

---

### 2. Données extraites : match Londrina vs CRB Maceio

Les informations extraites à l'instant précis de la capture d'écran ont été sauvegardées avec succès dans le fichier `1xbet_detailed_live.json`.

Voici le résumé complet des données réelles collectées à la minute **60:28** :

* **Score Actuel :** 3 - 0 (Mi-temps : 2-0 / $2^{\text{ème}}$ mi-temps en cours : 1-0)
* **Minute de jeu :** 60:28 ($2^{\text{ème}}$ mi-temps)
* **Lieu du match :** Vitorino Gonçalves Dias (Londrina)
* **Conditions Météo :** Ensoleillé, +22°C (Vent : 2.5 m/s, Humidité : 66%, Pression : 764 mmHg)

#### Statistiques du match (Londrina / CRB Maceio) :
* **xG (Buts attendus) :** 0.87 / 0.76
* **Attaques (Dangereuses) :** 35 (20) / 83 (44)
* **Possession de balle :** 33% / 67%
* **Tirs Cadrés (Non cadrés) :** 3 (1) / 4 (9)
* **Arrêts du gardien :** 4 / 0
* **Corners :** 2 / 7
* **Cartons Jaunes (Rouges) :** 2 (0) / 3 (0)
* **Pénaltys :** 1 / 0
* **Passes clés :** 3 / 8
* **Précision des passes :** 73% / 88%
* **Centres :** 9 / 18
* **Cotes 1X2 :** Verrouillées (Match suspendu / suspendu en raison de l'évolution du score).

### Entraînement sur page locale : Validation des sélecteurs
L'agent va lister ici chaque statistique trouvée avec son 'chemin' technique pour que vous puissiez confirmer l'exactitude.

In [14]:
mission_validation_technique = """
1. Examine le fichier 'londrina_local.html'.
2. Pour chaque statistique suivante, donne-moi la valeur extraite ET le sélecteur CSS précis utilisé :
   - Minute de jeu
   - Possession (Londrina vs CRB)
   - Tirs cadrés / non cadrés
   - xG (Expected Goals)
   - Cartons Jaunes
3. Si tu as trouvé ces données dans l'objet 'window.__BETTING_APP__', précise la clé JSON exacte.
4. Présente cela sous forme de tableau pour que l'utilisateur puisse valider.
"""

interaction_validation = client.interactions.create(
    agent=xbet_agent.id,
    input=mission_validation_technique,
    environment=interaction_correction.environment_id,
)

print("--- VÉRIFICATION TECHNIQUE DES DONNÉES (LONDINA) ---")
display(Markdown(interaction_validation.output_text))

--- VÉRIFICATION TECHNIQUE DES DONNÉES (LONDINA) ---


Voici l'analyse détaillée des données extraites du fichier local `londrina_local.html` pour la rencontre **Londrina vs CRB** (ID du match : `734051728`).

### Synthèse du raisonnement
Pour accomplir cette tâche, nous avons analysé la structure du DOM de la page ainsi que le script d'initialisation de l'application de paris. Nous avons localisé la liste des statistiques réelles dans le DOM (`.scoreboard-list`), identifié les classes CSS dynamiques générées par le framework Vue.js, puis extrait le payload JSON correspondant aux données brutes du match de football.

---

### Tableau d'extraction des statistiques

| Statistique | Valeur (Londrina vs CRB) | Sélecteur CSS précis utilisé | Clé JSON exacte (dans l'état de l'application) |
| :--- | :---: | :--- | :--- |
| **Minute de jeu** | **60:28** *(Affichage DOM)* <br> **53:12** *(Données JSON)* | `.ui-game-timer__time span.ui-caption` | `unparsedScoresData.TS` *(durée en secondes : 3192)* / `gameTimeStatus` |
| **Possession %** | **33% vs 67%** *(DOM)* <br> **32% vs 68%** *(JSON)* | `.scoreboard-stats-value--team-1` et `.scoreboard-stats-value--team-2` pour la ligne contenant `% de possession de balle` | `unparsedScoresData.ST[0].Value` -> Objet ayant `"ID": 29` (clés `"S1"` et `"S2"`) |
| **Tirs cadrés** | **3 vs 4** | `.scoreboard-stats-value--team-1` et `.scoreboard-stats-value--team-2` pour la ligne contenant `Tirs cadrés` | `unparsedScoresData.ST[0].Value` -> Objet ayant `"ID": 59` (clés `"S1"` et `"S2"`) |
| **Tirs non cadrés** | **1 vs 9** | `.scoreboard-stats-value--team-1` et `.scoreboard-stats-value--team-2` pour la ligne contenant `Tirs non cadrés` | `unparsedScoresData.ST[0].Value` -> Objet ayant `"ID": 60` (clés `"S1"` et `"S2"`) |
| **xG (Expected Goals)** | **0.87 vs 0.76** *(DOM)* <br> **0.87 vs 0.68** *(JSON)* | `.scoreboard-stats-value--team-1` et `.scoreboard-stats-value--team-2` pour la ligne contenant `xG` | `unparsedScoresData.ST[0].Value` -> Objet ayant `"ID": 93` (clés `"S1"` et `"S2"`) |
| **Cartons Jaunes** | **2 vs 3** | `.scoreboard-stats-value--team-1` et `.scoreboard-stats-value--team-2` pour la ligne contenant `Cartons jaunes` | `unparsedScoresData.ST[0].Value` -> Objet ayant `"ID": 26` (clés `"S1"` et `"S2"`) |

---

### Structure et chemin d'accès JSON dans `window.__BETTING_APP__`

Les statistiques d'avant-match et de live sont injectées via le serveur dans l'état global hydraté par l'application. La clé d'accès exacte contenant ces données de statistiques réelles dans l'objet global est la suivante :

```javascript
window.serverData.__BETTING_APP__.state.game.gamesById["734051728"].unparsedScoresData.ST[0].Value
```

Dans ce tableau d'objets `Value`, chaque statistique est identifiée par un code `"ID"` (par exemple `29` pour la possession, `93` pour les xG, `59` pour les tirs cadrés) avec les scores respectifs de l'équipe à domicile (Londrina) dans `"S1"` (ou `"firstOpponentScore"`) et de l'équipe à l'extérieur (CRB) dans `"S2"` (ou `"secondOpponentScore"`).

### Session d'entraînement approfondie (Page Hors-ligne)
L'agent doit maintenant identifier TOUTES les données disponibles sur la page de Londrina pour valider sa capacité de détection globale.

In [15]:
mission_apprentissage_exhaustif = """
1. Analyse à nouveau le fichier 'londrina_local.html' très minutieusement.
2. Liste TOUTES les statistiques disponibles (ex: attaques, attaques dangereuses, arrêts de gardien, corners, météo, xG, etc.).
3. Pour chaque donnée, indique :
   - Le libellé visible sur la page.
   - La valeur extraite.
   - Si elle vient du DOM (HTML) ou de l'objet JSON 'window.__BETTING_APP__'.
4. Si tu trouves des données 'cachées' (non visibles directement mais présentes dans le code), signale-les.
5. Produis un rapport détaillé pour validation par l'utilisateur.
"""

interaction_apprentissage = client.interactions.create(
    agent=xbet_agent.id,
    input=mission_apprentissage_exhaustif,
    environment=interaction_correction.environment_id,
)

print("--- RAPPORT D'APPRENTISSAGE EXHAUSTIF ---")
display(Markdown(interaction_apprentissage.output_text))

--- RAPPORT D'APPRENTISSAGE EXHAUSTIF ---


Voici un rapport d'analyse extrêmement minutieux du fichier `londrina_local.html`. Cette analyse met en lumière la structure technique de la page de 1XBET, combinant le **DOM HTML** (le rendu visuel mis à jour en temps réel par le client) et l'état de réhydratation initial stocké dans le script JS de l'application via l'objet **`window.__BETTING_APP__`** (géré côté serveur au moment du rendu initial).

---

# RAPPORT D'ANALYSE DE DONNÉES TEMPS RÉEL (LONDRINA VS CRB MACEIO)

## 1. Contexte & Architecture Technique de 1XBET
L'analyse approfondie du fichier révèle que 1XBET utilise une architecture de rendu hybride **SSR (Server-Side Rendering)** avec réhydratation **CSR (Client-Side Rendering)** :
*   **Rendu Serveur (SSR) :** Le serveur génère une page HTML figée contenant les données à la **minute 53:12** du match. Cet état est sérialisé sous forme de code JavaScript exécutable dans l'objet global `window.__BETTING_APP__` (injecté via un IIFE compresseur de variables dans le bloc de script 20).
*   **Rendu Navigateur (CSR) :** Une fois la page chargée dans le navigateur, l'application se connecte à un canal temps réel (probablement WebSockets ou requêtes HTTP à haute fréquence) pour mettre à jour l'interface utilisateur. Le DOM HTML capturé affiche ainsi le match à la **minute 60:28**.

Cette différence de **7 minutes et 16 secondes** de jeu permet d'observer l'évolution statistique exacte durant cet intervalle.

---

## 2. Table Comparative des Statistiques (Londrina vs CRB Maceio)

Le tableau suivant liste **toutes les statistiques de match disponibles**, en indiquant leur libellé, leurs valeurs respectives (DOM vs JSON), et leur provenance technique précise.

| Libellé visible sur la page | Valeur DOM (Client - `60:28`) | Valeur JSON (Serveur - `53:12`) | Source DOM (Sélecteur HTML / Classe) | Emplacement dans `window.__BETTING_APP__` (JSON) |
| :--- | :---: | :---: | :--- | :--- |
| **Minutes de jeu** | `60:28` | `53:12` (ou `"53 minutes"`) | `.scoreboard-section-block` | `unparsedScoresData.TS` (`3192` s) & `.timeFormatted` / `gameTimeStatus` |
| **Score** | `3 - 0` | `3 - 0` | `.scoreboard-intro__score` / `.scoreboard-team` | `firstOpponentFullScore: 3` / `secondOpponentFullScore: 0` |
| **Score par mi-temps** | `1ère: 2-0` / `2ème: 1-0` | `1ère: 2-0` / `2ème: 1-0` | `.scoreboard-section__scroll` (Texte brut) | `periodScores` (Tableau avec labels et scores) |
| **xG (Expected Goals)** | `0.87 vs 0.76` | `0.87 vs 0.68` | `.scoreboard-stats__row` / `.scoreboard-stats-value` | `summary["0"][0]` (id: 93) & `unparsedScoresData.ST[0].Value[0]` |
| **Attaques** | `35 vs 83` | `32 vs 78` | `.scoreboard-stats-table-view-name` (Attaques) | `summary["0"][1]` (id: 45) & `unparsedScoresData.ST[0].Value[1]` |
| **Attaques dangereuses** | `20 vs 44` | `18 vs 36` | `.scoreboard-stats-table-view-name` (Attaques d.) | `summary["0"][2]` (id: 58) & `unparsedScoresData.ST[0].Value[2]` |
| **% de possession de balle** | `33 vs 67` | `32 vs 68` | `.scoreboard-stats-table-view-name` (% poss.) | `summary["0"][3]` (id: 29) & `unparsedScoresData.ST[0].Value[3]` |
| **Tirs cadrés** | `3 vs 4` | `3 vs 4` | `.scoreboard-stats-table-view-name` (Tirs cad.) | `summary["0"][4]` (id: 59) & `unparsedScoresData.ST[0].Value[4]` |
| **Tirs non cadrés** | `1 vs 9` | `1 vs 7` | `.scoreboard-stats-table-view-name` (Tirs n. cad) | `summary["0"][5]` (id: 60) & `unparsedScoresData.ST[0].Value[5]` |
| **Cartons jaunes** | `2 vs 3` | `2 vs 3` | `.scoreboard-fouls__item--yellow-cards` | `summary["0"][6]` (id: 26) & `statistics.firstTeamFoulStatistics.yellowCardsCount` |
| **Arrêts (du gardien)** | `4 vs 0` | `4 vs 0` | `.scoreboard-stats-table-view-name` (Arrêts) | `summary["0"][7]` (id: 47) & `unparsedScoresData.ST[0].Value[7]` |
| **Corners** | `2 vs 7` | `2 vs 5` | `.scoreboard-fouls__item--corners` | `summary["0"][8]` (id: 70) & `statistics.firstTeamFoulStatistics.cornerKicksCount` |
| **Cartons rouges** | `0 vs 0` | `0 vs 0` | `.scoreboard-stats-table-view-name` (C. rouges) | `summary["0"][9]` (id: 71) & `statistics.firstTeamFoulStatistics.redCardsCount` |
| **Pénalty** | `1 vs 0` | `1 vs 0` | `.scoreboard-fouls__item--penalty` | `summary["0"][10]` (id: 72) & `statistics.firstTeamFoulStatistics.penaltyKicksCount` |
| **Remplaçants (Substitutions)** | `1 vs 4` | `1 vs 0` | `.scoreboard-stats-table-view-name` (Remplaçants) | `summary["0"][11]` (id: 92) & `unparsedScoresData.ST[0].Value[11]` |
| **Passes clés** | `3 vs 8` | `3 vs 7` | `.scoreboard-stats-table-view-name` (Passes clés) | `summary["0"][12]` (id: 94) & `unparsedScoresData.ST[0].Value[12]` |
| **Précision des passes %** | `73 vs 88` | `73 vs 88` | `.scoreboard-stats-table-view-name` (Précis. %) | `summary["0"][13]` (id: 95) & `unparsedScoresData.ST[0].Value[13]` |
| **Centres** | `9 vs 18` | `8 vs 15` | `.scoreboard-stats-table-view-name` (Centres) | `summary["0"][14]` (id: 96) & `unparsedScoresData.ST[0].Value[14]` |

---

## 3. Données de Stade, Ligue & Conditions Météo

| Donnée | Valeur DOM (Client - `60:28`) | Valeur JSON (Serveur - `53:12`) | Source technique (DOM / JSON) |
| :--- | :--- | :--- | :--- |
| **Ligue / Compétition** | `Brésil. Tour 16` | `Brazil. Campeonato Brasileiro. Serie B` | DOM: `.scoreboard-header` / JSON: `champNameEng` / `champName` |
| **Nom du Stade** | `Vitorino Gonçalves Dias (Londrina)` | `Vitorino Gonçalves Dias (Londrina)` | DOM: `.scoreboard-stadium` / JSON: `matchInfo.stadium` |
| **Ville / Pays** | Non visible | `Londrina` / `Brésil` | JSON uniquement : `matchInfo.city` & `matchInfo.country` |
| **Température** | `+22°C` | `+24°C` | DOM: `.scoreboard-weather` (Aria) / JSON: `weatherData.temperature` |
| **Condition Météo** | `Ensoleillé` | `Ensoleillé` | DOM: `.scoreboard-weather` (Aria) / JSON: `weatherData.weatherCondition` |
| **Vent** | `2.5 Vent N, m/sec` | `2.5 m/sec` (Direction: `Vent N, m/sec`) | DOM: `.scoreboard-weather` (Aria) / JSON: `weatherData.windSpeedWithDimension` |
| **Pression Atmosphérique** | `765 Pression, mmHg` | `764 mmHg` | DOM: `.scoreboard-weather` (Aria) / JSON: `weatherData.pressureWithDimension` |
| **Humidité** | `72 Humidité, %` | `66 %` | DOM: `.scoreboard-weather` (Aria) / JSON: `weatherData.humidityWithDimension` |

*Note de l'expert :* On constate un rafraîchissement météorologique de **-2°C** et une augmentation d'humidité de **+6%** en l'espace de 7 minutes entre le snapshot serveur (`53:12`) et l'affichage temps réel (`60:28`).

---

## 4. Découverte de Données Cachées (Invisibles dans l'UI)

L'extraction minutieuse de l'objet de hydration `window.__BETTING_APP__` a permis de découvrir plusieurs flux de données stratégiques totalement invisibles à un utilisateur consultant la page :

### A. Données Multi-Matchs en Arrière-Plan (Hidden Live Stream)
Bien que l'utilisateur soit sur la page exclusive de **Londrina vs CRB Maceio**, l'objet `normalizedDataItems-betting/home/live/dashboardStore-games` contient les données statistiques complètes en temps réel de **9 autres matchs** en cours !

Voici la liste de ces événements "fantômes" présents dans le code de la page :
1.  **Tennis (Wimbledon) :**
    *   **Frances Tiafoe vs Alexander Bublik** (ID: `734031204`) - *Stats incluses : Aces (7 vs 32), Fautes (6 vs 4), Points de break (14% vs 0%), % 1er service (61% vs 66%).*
    *   **Grigor Dimitrov vs Matteo Berrettini** (ID: `734034318`) - *Stats incluses : Aces (13 vs 19), Double fautes (4 vs 3), Breaks (3 vs 2).*
2.  **Basket-ball (NBA Summer League) :**
    *   **Bucks de Milwaukee vs Golden State Warriors Blue** (ID: `734051772`) - *Score : 71-58. Stats incluses : Fautes (8 vs 11), Paniers à 2 pts (26 vs 18), Paniers à 3 pts (6 vs 7), Lancers francs % (50% vs 50%).*
    *   **Grizzlies de Memphis vs Oklahoma City Thunder** (ID: `734051740`) - *Score : 81-52. Stats incluses : Paniers à 3 pts (13 vs 5).*
3.  **Football (Autres ligues) :**
    *   **Indy Eleven vs Battery de Charleston** (ID: `734064775` - USL US) - *Score : 0-0 (10:31). Stats incluses : Possession (19% vs 81%), Attaques (6 vs 12).*
    *   **Criciúma vs Sport Club do Recife** (ID: `734051777` - Série B Brésil) - *Score : 1-0 (48:43). Stats incluses : xG (0.82 vs 0.08), Tirs (8 vs 3), Corners (1 vs 2).*
    *   **Novorizontino vs Atlético Clube Goianiense** (ID: `734051851` - Série B Brésil) - *Score : 1-0 (58:08). Stats de match complètes incluses.*
    *   **Libertad Loja vs Leones Del Norte** (ID: `734051858` - Équateur) - *Score : 1-2 (50:08). Stats incluses : xG (0.10 vs 1.83), Cartons rouges (1 vs 0), Pénaltys (0 vs 1).*
    *   **Longford Town vs University College Dublin** (ID: `734047930` - Irlande) - *Score : 2-1 (85:41). Stats complètes incluses.*

### B. Probabilités de Victoire Algorithmiques (Winning Probabilities)
Le bookmaker calcule en temps réel ses probabilités internes de résultat. Ces pourcentages de probabilités, souvent absents de l'UI en direct pour pousser aux cotes dynamiques, sont pourtant sérialisés dans le JSON.
*   **Criciúma vs Sport Club :** Victoire Criciuma = **48%**, Nul = **19%**, Victoire Sport Club = **33%** (chemin : `winningProbabilities`).
*   **Grizzlies vs OKC (Basket) :** Victoire Grizzlies = **55%**, Victoire OKC = **45%**.
*   **Libertad Loja vs Leones :** Victoire Libertad = **13%**, Nul = **12%**, Victoire Leones = **75%**.

### C. Identifiants de Fournisseurs Tiers
La clé de liaison pour appeler les widgets de statistiques graphiques animés est présente :
*   `statisticGameId`: `"6997cde75e99bd05c6e9542f"`
*   `stageId`: `"69947ea7785fcb8116a5d421"`
Ces IDs permettent de bypasser l'interface 1XBET et d'effectuer des requêtes directes vers le fournisseur de données de statistiques (ex: Sportradar ou Betradar).

---

## 5. Synthèse & Validation pour l'Extracteur
L'analyse minutieuse confirme que la création d'un script d'extraction robuste (tel que `parser.py`) doit **prioritairement cibler l'objet JSON `window.__BETTING_APP__`** plutôt que le DOM HTML pour les raisons suivantes :
1.  **Fiabilité structurelle :** Les classes CSS du DOM (ex: `scoreboard-stats-table-view-row`) sont générées à l'aide d'obfuscations dynamiques de frameworks (comme `data-v-fd7badb7`), sujettes à de fréquentes cassures lors des mises à jour du site.
2.  **Richesse de l'information :** Le JSON fournit des métadonnées impossibles à obtenir via le DOM (Database IDs des équipes/ligues, fichiers logo CDN comme `45393.webp`, probabilités implicites, etc.).
3.  **Performances et Extensibilité :** En une seule capture de la page de hydration, vous récupérez non seulement les détails du match cible, mais également l'état statistique complet de **tous les autres sports** (Tennis, Basket) s'exécutant simultanément (répondant parfaitement au besoin d'extension vers le Basket-ball formulé dans le protocole d'extraction).

### Questions pour affiner l'agent :
- Quelles **catégories sportives** spécifiques (Football, Tennis, etc.) ?
- Quel est le **format de sortie** souhaité (un fichier JSON simple ou une analyse comparative) ?
- Y a-t-il des **horaires spécifiques** pour la collecte (données en direct / live) ?

je veux les matchs football en live, ce que tu propose n'est pas mal, je vais pour l'instant me concenter sur les matchs en live dans le football et je peux après intéger le basketball dedans. Il faut aussi ajouter la minuite de jeu à laquelle le jeu est. il doit pouvoir ouvrir la page des statistiques et collecter les données disponibles de chaques équipes , score, carton, possession, tirs cadrés et non cadrés dans le football et paniers tentés dans le basketball et tout les autres statistiques nécéssaire et disponibles, courbe de variation des côtes si disponible, statistiques d'avant-matchs disponibles et autres choses nécéssaires. si tu rencontre un problème, je vais t'apprendre, aprèes je peux peut-être aller sur les avant-matchs pour collecter des données mais pour l'instant ce sera les matchs en live et surtout le football en priorité.